In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

players = pd.read_csv(RAW_DIR / "players.csv")
valuations = pd.read_csv(RAW_DIR / "player_valuations.csv")
appearances = pd.read_csv(RAW_DIR / "appearances.csv")
clubs = pd.read_csv(RAW_DIR / "clubs.csv")
competitions = pd.read_csv(RAW_DIR / "competitions.csv")

print("Players:", players.shape)
print("Valuations:", valuations.shape)
print("Appearances:", appearances.shape)
print("Clubs:", clubs.shape)
print("Competitions:", competitions.shape)

Players: (47702, 26)
Valuations: (616377, 6)
Appearances: (1862208, 13)
Clubs: (796, 17)
Competitions: (67, 11)


In [2]:
print("\nPLAYERS:\n",players.columns.tolist())
print("\nVALUATIONS:\n",players.columns.tolist())
print("\nAPPEARANCES:\n",players.columns.tolist())



PLAYERS:
 ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

VALUATIONS:
 ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

APPEARANCES:
 

In [3]:
appearances.head(1)

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90


In [4]:
player_stats=appearances.groupby("player_id").agg({
    "minutes_played":"sum",
    "goals":"sum",
    "assists":"sum"
}).reset_index()

player_stats.head(10)

,player_id,minutes_played,goals,assists
0,10,8808,48,25
1,26,13508,0,0
2,65,8788,38,13
3,77,307,0,0
4,80,1080,0,0
5,109,3584,1,2
6,123,427,0,1
7,132,3987,9,4
8,215,6038,26,8
9,258,1075,0,2


In [5]:
valuations_sorted = valuations.sort_values("date", ascending=False)

latest_valuations = valuations_sorted.drop_duplicates(subset=["player_id"])

latest_valuations = latest_valuations[["player_id", "market_value_in_eur"]]

latest_valuations.head()

,player_id,market_value_in_eur
616376,1519157,100000
616284,668731,700000
616290,701224,2200000
616289,688019,800000
616288,685657,2000000


In [6]:
player_info = players[[
    "player_id",
    "name",
    "date_of_birth",
    "position",
    "current_club_id"
]]

player_info.head()

,player_id,name,date_of_birth,position,current_club_id
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,398
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,16
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,1091
3,77,Lúcio,1978-05-08 00:00:00,Defender,506
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,27


In [7]:
final_df=player_info.merge(player_stats,on='player_id',how='inner')
final_df=final_df.merge(latest_valuations,on='player_id',how='inner')
final_df.head()

,player_id,name,date_of_birth,position,current_club_id,minutes_played,goals,assists,market_value_in_eur
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,398,8808,48,25,1000000
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,16,13508,0,0,750000
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,1091,8788,38,13,1000000
3,77,Lúcio,1978-05-08 00:00:00,Defender,506,307,0,0,200000
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,27,1080,0,0,100000


In [8]:
final_df['date_of_birth']=pd.to_datetime(final_df['date_of_birth'])
final_df["age"]=(pd.Timestamp("today")-final_df["date_of_birth"]).dt.days


In [9]:
final_df["age"] = (pd.Timestamp("today") - final_df["date_of_birth"]).dt.days / 365
final_df = final_df.dropna(subset=["age"])
final_df["age"] = final_df["age"].astype(int)

In [10]:
final_df=final_df[final_df["minutes_played"]>0]

In [11]:
final_df["goals_per_90"] = final_df["goals"] / (final_df["minutes_played"] / 90)

final_df["assists_per_90"] = final_df["assists"] / (final_df["minutes_played"] / 90)

In [12]:
final_df["goal_contributions_per_90"] = (
    final_df["goals_per_90"] + final_df["assists_per_90"]
)

In [13]:
import numpy as np

final_df["log_market_value"] = np.log1p(final_df["market_value_in_eur"])

In [14]:
final_df[[
    "name",
    "age",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "market_value_in_eur",
    "log_market_value"
]].head()

,name,age,goals_per_90,assists_per_90,goal_contributions_per_90,market_value_in_eur,log_market_value
0,Miroslav Klose,47,0.490463,0.255450,0.745913,1000000,13.815512
1,Roman Weidenfeller,45,0.000000,0.000000,0.000000,750000,13.527830
2,Dimitar Berbatov,45,0.389167,0.133136,0.522303,1000000,13.815512
3,Lúcio,48,0.000000,0.000000,0.000000,200000,12.206078
4,Tom Starke,45,0.000000,0.000000,0.000000,100000,11.512935


In [15]:
features_df = final_df[[
    "age",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "position",
    "log_market_value"
]].copy()

features_df.head()

,age,goals_per_90,assists_per_90,goal_contributions_per_90,position,log_market_value
0,47,0.490463,0.255450,0.745913,Attack,13.815512
1,45,0.000000,0.000000,0.000000,Goalkeeper,13.527830
2,45,0.389167,0.133136,0.522303,Attack,13.815512
3,48,0.000000,0.000000,0.000000,Defender,12.206078
4,45,0.000000,0.000000,0.000000,Goalkeeper,11.512935


In [16]:
features_df = final_df[[
    "age",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "position",
    "log_market_value"
]].copy()

features_df.head()

,age,goals_per_90,assists_per_90,goal_contributions_per_90,position,log_market_value
0,47,0.490463,0.255450,0.745913,Attack,13.815512
1,45,0.000000,0.000000,0.000000,Goalkeeper,13.527830
2,45,0.389167,0.133136,0.522303,Attack,13.815512
3,48,0.000000,0.000000,0.000000,Defender,12.206078
4,45,0.000000,0.000000,0.000000,Goalkeeper,11.512935


In [17]:
features_df = pd.get_dummies(features_df, columns=["position"], drop_first=True)


In [18]:
X = features_df.drop("log_market_value", axis=1)
y = features_df["log_market_value"]

print(X.shape)
print(y.shape)


(27620, 8)
(27620,)


In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)
print(X_train.shape,X_test.shape)

(22096, 8) (5524, 8)


In [20]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [21]:
y_pred = model.predict(X_test)

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 1.1016654768912642
RMSE: 1.4339111242549436
R2 Score: 0.15992226824820643


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R2 Score:", rf_r2)

Random Forest MAE: 0.9849321994036965
Random Forest RMSE: 1.2642344000563517
Random Forest R2 Score: 0.3469743928803747


In [24]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting MAE:", gb_mae)
print("Gradient Boosting RMSE:", gb_rmse)
print("Gradient Boosting R2 Score:", gb_r2)

Gradient Boosting MAE: 0.9355078234305274
Gradient Boosting RMSE: 1.2038249374398522
Gradient Boosting R2 Score: 0.4078909814469396


In [25]:
features_df_v2 = final_df[[
    "age",
    "minutes_played",
    "goals",
    "assists",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "position",
    "log_market_value"
]].copy()

features_df_v2.head()

,age,minutes_played,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,position,log_market_value
0,47,8808,48,25,0.490463,0.255450,0.745913,Attack,13.815512
1,45,13508,0,0,0.000000,0.000000,0.000000,Goalkeeper,13.527830
2,45,8788,38,13,0.389167,0.133136,0.522303,Attack,13.815512
3,48,307,0,0,0.000000,0.000000,0.000000,Defender,12.206078
4,45,1080,0,0,0.000000,0.000000,0.000000,Goalkeeper,11.512935


In [26]:
features_df_v2 = pd.get_dummies(
    features_df_v2,
    columns=["position"],
    drop_first=True
)

features_df_v2.head()

,age,minutes_played,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,log_market_value,position_Defender,position_Goalkeeper,position_Midfield,position_Missing
0,47,8808,48,25,0.490463,0.255450,0.745913,13.815512,False,False,False,False
1,45,13508,0,0,0.000000,0.000000,0.000000,13.527830,False,True,False,False
2,45,8788,38,13,0.389167,0.133136,0.522303,13.815512,False,False,False,False
3,48,307,0,0,0.000000,0.000000,0.000000,12.206078,True,False,False,False
4,45,1080,0,0,0.000000,0.000000,0.000000,11.512935,False,True,False,False


In [27]:
X_v2 = features_df_v2.drop("log_market_value", axis=1)
y_v2 = features_df_v2["log_market_value"]

print("X_v2 shape:", X_v2.shape)
print("y_v2 shape:", y_v2.shape)

X_v2 shape: (27620, 11)
y_v2 shape: (27620,)


In [28]:
from sklearn.model_selection import train_test_split

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2,
    y_v2,
    test_size=0.2,
    random_state=42
)

print(X_train_v2.shape, X_test_v2.shape)

(22096, 11) (5524, 11)


In [29]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

gb_model_v2 = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model_v2.fit(X_train_v2, y_train_v2)

gb_pred_v2 = gb_model_v2.predict(X_test_v2)

gb_mae_v2 = mean_absolute_error(y_test_v2, gb_pred_v2)
gb_rmse_v2 = np.sqrt(mean_squared_error(y_test_v2, gb_pred_v2))
gb_r2_v2 = r2_score(y_test_v2, gb_pred_v2)

print("Gradient Boosting V2 MAE:", gb_mae_v2)
print("Gradient Boosting V2 RMSE:", gb_rmse_v2)
print("Gradient Boosting V2 R2 Score:", gb_r2_v2)

Gradient Boosting V2 MAE: 0.8086562828798686
Gradient Boosting V2 RMSE: 1.0332243821648868
Gradient Boosting V2 R2 Score: 0.5638214572896523


In [30]:
club_strength = final_df.groupby("current_club_id")["market_value_in_eur"].mean().reset_index()

club_strength = club_strength.rename(
    columns={"market_value_in_eur": "club_avg_market_value"}
)

final_df = final_df.merge(club_strength, on="current_club_id", how="left")

final_df["log_club_avg_market_value"] = np.log1p(final_df["club_avg_market_value"])

final_df[["name", "current_club_id", "club_avg_market_value", "log_club_avg_market_value"]].head()

,name,current_club_id,club_avg_market_value,log_club_avg_market_value
0,Miroslav Klose,398,3.366143e+06,15.029278
1,Roman Weidenfeller,16,1.164602e+07,16.270475
2,Dimitar Berbatov,1091,1.412349e+06,14.160766
3,Lúcio,506,1.175469e+07,16.279763
4,Tom Starke,27,2.908456e+07,17.185718


In [31]:
features_df_v3 = final_df[[
    "age",
    "minutes_played",
    "goals",
    "assists",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "log_club_avg_market_value",
    "position",
    "log_market_value"
]].copy()

features_df_v3 = pd.get_dummies(
    features_df_v3,
    columns=["position"],
    drop_first=True
)

features_df_v3.head()

,age,minutes_played,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,log_club_avg_market_value,log_market_value,position_Defender,position_Goalkeeper,position_Midfield,position_Missing
0,47,8808,48,25,0.490463,0.255450,0.745913,15.029278,13.815512,False,False,False,False
1,45,13508,0,0,0.000000,0.000000,0.000000,16.270475,13.527830,False,True,False,False
2,45,8788,38,13,0.389167,0.133136,0.522303,14.160766,13.815512,False,False,False,False
3,48,307,0,0,0.000000,0.000000,0.000000,16.279763,12.206078,True,False,False,False
4,45,1080,0,0,0.000000,0.000000,0.000000,17.185718,11.512935,False,True,False,False


In [32]:
X_v3 = features_df_v3.drop("log_market_value", axis=1)
y_v3 = features_df_v3["log_market_value"]

print("X_v3 shape:", X_v3.shape)
print("y_v3 shape:", y_v3.shape)

X_v3 shape: (27620, 12)
y_v3 shape: (27620,)


In [33]:
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3,
    y_v3,
    test_size=0.2,
    random_state=42
)

gb_model_v3 = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model_v3.fit(X_train_v3, y_train_v3)

gb_pred_v3 = gb_model_v3.predict(X_test_v3)

gb_mae_v3 = mean_absolute_error(y_test_v3, gb_pred_v3)
gb_rmse_v3 = np.sqrt(mean_squared_error(y_test_v3, gb_pred_v3))
gb_r2_v3 = r2_score(y_test_v3, gb_pred_v3)

print("Gradient Boosting V3 MAE:", gb_mae_v3)
print("Gradient Boosting V3 RMSE:", gb_rmse_v3)
print("Gradient Boosting V3 R2 Score:", gb_r2_v3)

Gradient Boosting V3 MAE: 0.6490435439644178
Gradient Boosting V3 RMSE: 0.841488999214051
Gradient Boosting V3 R2 Score: 0.7106843740967954


In [34]:
feature_importance = pd.DataFrame({
    "feature": X_v3.columns,
    "importance": gb_model_v3.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance

,feature,importance
7,log_club_avg_market_value,0.452243
0,age,0.250620
1,minutes_played,0.243015
2,goals,0.023909
6,goal_contributions_per_90,0.014147
3,assists,0.007105
4,goals_per_90,0.005973
9,position_Goalkeeper,0.001815
5,assists_per_90,0.000949
10,position_Midfield,0.000181


In [35]:
results_df = final_df.loc[X_test_v3.index].copy()

results_df["predicted_log_market_value"] = gb_pred_v3

results_df["predicted_market_value_eur"] = np.expm1(
    results_df["predicted_log_market_value"]
)

results_df["actual_market_value_eur"] = results_df["market_value_in_eur"]

results_df["value_gap_eur"] = (
    results_df["predicted_market_value_eur"] - results_df["actual_market_value_eur"]
)

results_df[[
    "name",
    "age",
    "position",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur"
]].head()

,name,age,position,actual_market_value_eur,predicted_market_value_eur,value_gap_eur
24781,Ismaël Doukouré,22,Defender,20000000,1.562763e+07,-4.372368e+06
4404,Egor Filipenko,38,Defender,150000,1.471996e+05,-2.800389e+03
25239,Steven Nsimba,29,Attack,800000,3.789428e+05,-4.210572e+05
21216,Marcus Lindberg,25,Midfield,700000,7.230435e+05,2.304351e+04
7763,Edin Visca,36,Attack,300000,1.170771e+06,8.707714e+05


In [36]:
undervalued_players = results_df[
    (results_df["age"] <= 25) &
    (results_df["minutes_played"] >= 900) &
    (results_df["value_gap_eur"] > 0) &
    (results_df["actual_market_value_eur"] >= 100000)
].copy()

undervalued_players = undervalued_players.sort_values(
    by="value_gap_eur",
    ascending=False
)

undervalued_players[[
    "name",
    "age",
    "position",
    "minutes_played",
    "goals",
    "assists",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur"
]].head(20)

,name,age,position,minutes_played,goals,assists,actual_market_value_eur,predicted_market_value_eur,value_gap_eur
21365,Gonçalo Ramos,24,Attack,10117,73,23,35000000,7.478560e+07,3.978560e+07
24824,Roony Bardghji,20,Attack,2714,14,4,15000000,4.144691e+07,2.644691e+07
24546,Savinho,22,Attack,7569,17,27,40000000,6.259201e+07,2.259201e+07
23121,Issa Kaboré,24,Defender,7433,1,9,4000000,2.569802e+07,2.169802e+07
18536,Rodrygo,25,Attack,17216,70,58,50000000,7.032666e+07,2.032666e+07
21466,Kang-in Lee,25,Midfield,12859,25,29,28000000,4.698373e+07,1.898373e+07
21395,Santiago Gimenez,25,Attack,8714,72,19,20000000,3.453081e+07,1.453081e+07
26336,Artem Smolyakov,22,Defender,5466,4,3,1800000,1.353965e+07,1.173965e+07
21848,Rayan Aït-Nouri,24,Defender,14455,11,26,40000000,5.034061e+07,1.034061e+07
23354,David Møller Wolfe,24,Defender,7652,6,12,10000000,2.031802e+07,1.031802e+07


In [37]:
undervalued_players["scouting_score"] = (
    0.45 * undervalued_players["value_gap_eur"].rank(pct=True) +
    0.25 * undervalued_players["goal_contributions_per_90"].rank(pct=True) +
    0.20 * undervalued_players["minutes_played"].rank(pct=True) +
    0.10 * (1 - undervalued_players["age"].rank(pct=True))
)

undervalued_players = undervalued_players.sort_values(
    by="scouting_score",
    ascending=False
)

undervalued_players[[
    "name",
    "age",
    "position",
    "minutes_played",
    "goals",
    "assists",
    "goal_contributions_per_90",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur",
    "scouting_score"
]].head(20)

,name,age,position,minutes_played,goals,assists,goal_contributions_per_90,actual_market_value_eur,predicted_market_value_eur,value_gap_eur,scouting_score
24546,Savinho,22,Attack,7569,17,27,0.523187,40000000,6.259201e+07,2.259201e+07,0.934948
21365,Gonçalo Ramos,24,Attack,10117,73,23,0.854008,35000000,7.478560e+07,3.978560e+07,0.930623
21440,Arsen Zakharyan,22,Midfield,9419,22,28,0.477758,10000000,1.737919e+07,7.379188e+06,0.913495
18536,Rodrygo,25,Attack,17216,70,58,0.669145,50000000,7.032666e+07,2.032666e+07,0.903633
21395,Santiago Gimenez,25,Attack,8714,72,19,0.939867,20000000,3.453081e+07,1.453081e+07,0.892388
22134,Sem Steijn,24,Midfield,9664,61,17,0.726407,11000000,1.696652e+07,5.966516e+06,0.891869
24824,Roony Bardghji,20,Attack,2714,14,4,0.596905,15000000,4.144691e+07,2.644691e+07,0.889965
26280,Clement Bischoff,20,Attack,2994,3,13,0.480962,5000000,1.376778e+07,8.767778e+06,0.859862
21466,Kang-in Lee,25,Midfield,12859,25,29,0.377945,28000000,4.698373e+07,1.898373e+07,0.857093
22145,Bjorn Meijer,23,Defender,7672,8,14,0.258081,3500000,1.349758e+07,9.997577e+06,0.842907


In [38]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(PROCESSED_DIR / "model_predictions.csv", index=False)
undervalued_players.to_csv(PROCESSED_DIR / "undervalued_players.csv", index=False)
feature_importance.to_csv(PROCESSED_DIR / "feature_importance.csv", index=False)

model_comparison = pd.DataFrame({
    "model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting V1",
        "Gradient Boosting V2",
        "Gradient Boosting V3"
    ],
    "mae": [
        mae,
        rf_mae,
        gb_mae,
        gb_mae_v2,
        gb_mae_v3
    ],
    "rmse": [
        rmse,
        rf_rmse,
        gb_rmse,
        gb_rmse_v2,
        gb_rmse_v3
    ],
    "r2_score": [
        r2,
        rf_r2,
        gb_r2,
        gb_r2_v2,
        gb_r2_v3
    ]
})

model_comparison.to_csv(PROCESSED_DIR / "model_comparison.csv", index=False)

model_comparison

,model,mae,rmse,r2_score
0,Linear Regression,1.101665,1.433911,0.159922
1,Random Forest,0.984932,1.264234,0.346974
2,Gradient Boosting V1,0.935508,1.203825,0.407891
3,Gradient Boosting V2,0.808656,1.033224,0.563821
4,Gradient Boosting V3,0.649044,0.841489,0.710684


In [39]:
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/processed/football_scouting.db")

conn = sqlite3.connect(DB_PATH)

players_model_data = final_df[[
    "player_id",
    "name",
    "age",
    "position",
    "current_club_id",
    "minutes_played",
    "goals",
    "assists",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "market_value_in_eur",
    "log_market_value",
    "club_avg_market_value",
    "log_club_avg_market_value"
]].copy()

players_model_data.to_sql(
    "players_model_data",
    conn,
    if_exists="replace",
    index=False
)

results_df.to_sql(
    "model_predictions",
    conn,
    if_exists="replace",
    index=False
)

undervalued_players.to_sql(
    "undervalued_players",
    conn,
    if_exists="replace",
    index=False
)

feature_importance.to_sql(
    "feature_importance",
    conn,
    if_exists="replace",
    index=False
)

model_comparison.to_sql(
    "model_comparison",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("SQLite database created successfully:", DB_PATH)

SQLite database created successfully: ..\data\processed\football_scouting.db


In [40]:
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

conn.close()

tables

,name
0,players_model_data
1,model_predictions
2,undervalued_players
3,feature_importance
4,model_comparison


In [41]:
conn = sqlite3.connect(DB_PATH)

best_model_query = """
SELECT 
    model,
    mae,
    rmse,
    r2_score
FROM model_comparison
ORDER BY r2_score DESC
LIMIT 1;
"""

best_model = pd.read_sql_query(best_model_query, conn)

best_model

,model,mae,rmse,r2_score
0,Gradient Boosting V3,0.649044,0.841489,0.710684


In [42]:
top_undervalued_query = """
SELECT
    name,
    age,
    position,
    minutes_played,
    goals,
    assists,
    actual_market_value_eur,
    predicted_market_value_eur,
    value_gap_eur,
    scouting_score
FROM undervalued_players
ORDER BY scouting_score DESC
LIMIT 10;
"""

top_undervalued_sql = pd.read_sql_query(top_undervalued_query, conn)

top_undervalued_sql

,name,age,position,minutes_played,goals,assists,actual_market_value_eur,predicted_market_value_eur,value_gap_eur,scouting_score
0,Savinho,22,Attack,7569,17,27,40000000,6.259201e+07,2.259201e+07,0.934948
1,Gonçalo Ramos,24,Attack,10117,73,23,35000000,7.478560e+07,3.978560e+07,0.930623
2,Arsen Zakharyan,22,Midfield,9419,22,28,10000000,1.737919e+07,7.379188e+06,0.913495
3,Rodrygo,25,Attack,17216,70,58,50000000,7.032666e+07,2.032666e+07,0.903633
4,Santiago Gimenez,25,Attack,8714,72,19,20000000,3.453081e+07,1.453081e+07,0.892388
5,Sem Steijn,24,Midfield,9664,61,17,11000000,1.696652e+07,5.966516e+06,0.891869
6,Roony Bardghji,20,Attack,2714,14,4,15000000,4.144691e+07,2.644691e+07,0.889965
7,Clement Bischoff,20,Attack,2994,3,13,5000000,1.376778e+07,8.767778e+06,0.859862
8,Kang-in Lee,25,Midfield,12859,25,29,28000000,4.698373e+07,1.898373e+07,0.857093
9,Bjorn Meijer,23,Defender,7672,8,14,3500000,1.349758e+07,9.997577e+06,0.842907


In [43]:
feature_importance_query = """
SELECT
    feature,
    importance
FROM feature_importance
ORDER BY importance DESC
LIMIT 10;
"""

top_features_sql = pd.read_sql_query(feature_importance_query, conn)

top_features_sql

,feature,importance
0,log_club_avg_market_value,0.452243
1,age,0.250620
2,minutes_played,0.243015
3,goals,0.023909
4,goal_contributions_per_90,0.014147
5,assists,0.007105
6,goals_per_90,0.005973
7,position_Goalkeeper,0.001815
8,assists_per_90,0.000949
9,position_Midfield,0.000181


In [44]:
young_attackers_query = """
SELECT
    name,
    age,
    position,
    goals,
    assists,
    goal_contributions_per_90,
    actual_market_value_eur,
    predicted_market_value_eur,
    value_gap_eur,
    scouting_score
FROM undervalued_players
WHERE age <= 23
  AND position = 'Attack'
ORDER BY scouting_score DESC
LIMIT 10;
"""

young_attackers_sql = pd.read_sql_query(young_attackers_query, conn)

young_attackers_sql

,name,age,position,goals,assists,goal_contributions_per_90,actual_market_value_eur,predicted_market_value_eur,value_gap_eur,scouting_score
0,Savinho,22,Attack,17,27,0.523187,40000000,6.259201e+07,2.259201e+07,0.934948
1,Roony Bardghji,20,Attack,14,4,0.596905,15000000,4.144691e+07,2.644691e+07,0.889965
2,Clement Bischoff,20,Attack,3,13,0.480962,5000000,1.376778e+07,8.767778e+06,0.859862
3,Eliesse Ben Seghir,21,Attack,14,8,0.446046,20000000,2.406559e+07,4.065593e+06,0.835467
4,Lasso Coulibaly,23,Attack,6,12,0.444079,900000,5.388033e+06,4.488033e+06,0.802076
5,Hyun-jun Yang,23,Attack,13,10,0.447181,3000000,6.109775e+06,3.109775e+06,0.796540
6,Mamadou Diakhon,20,Attack,4,10,0.566292,9000000,1.259167e+07,3.591672e+06,0.794810
7,Filip Bundgaard,21,Attack,23,7,0.504296,2500000,4.141830e+06,1.641830e+06,0.786332
8,Gabri Martínez,23,Attack,9,11,0.451354,5000000,7.257849e+06,2.257849e+06,0.764360
9,Gianluca Prestianni,20,Attack,3,4,0.318504,12000000,1.710225e+07,5.102251e+06,0.754844


In [45]:
conn.close()

In [1]:

































import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

clubs = pd.read_csv(RAW_DIR / "clubs.csv")
competitions = pd.read_csv(RAW_DIR / "competitions.csv")

results_df = pd.read_csv(PROCESSED_DIR / "model_predictions.csv")
undervalued_players = pd.read_csv(PROCESSED_DIR / "undervalued_players.csv")
feature_importance = pd.read_csv(PROCESSED_DIR / "feature_importance.csv")
model_comparison = pd.read_csv(PROCESSED_DIR / "model_comparison.csv")

In [2]:
print("CLUB COLUMNS:")
print(clubs.columns.tolist())

print("COMPITITION COLUMNS:")
print(competitions.columns.tolist())

CLUB COLUMNS:
['club_id', 'club_code', 'name', 'domestic_competition_id', 'total_market_value', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_name', 'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season', 'filename', 'url']
COMPITITION COLUMNS:
['competition_id', 'competition_code', 'name', 'sub_type', 'type', 'country_id', 'country_name', 'domestic_league_code', 'confederation', 'total_clubs', 'url']


In [3]:
clubs.head()

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...
1,10004,paris-fc,Paris Football Club,FR1,NaN,31,28.5,17,54.8,8,Stade Jean Bouin,19904,€-72.30m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/paris-fc/start...
2,10010,esporte-clube-bahia,Esporte Clube Bahia,BRA1,NaN,32,26.2,6,18.8,3,Arena Fonte Nova,47364,+€8.14m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/esporte-clube-...
3,1003,leicester-city,Leicester City,GB1,NaN,29,25.9,17,58.6,9,King Power Stadium,32259,+€57.30m,Steve Cooper,2024,../data/raw/transfermarkt-scraper/2024/clubs.j...,https://www.transfermarkt.co.uk/leicester-city...
4,1005,us-lecce,Unione Sportiva Lecce,IT1,NaN,27,25.1,23,85.2,10,Ettore Giardiniero,31559,+€8.62m,NaN,2025,../data/raw/transfermarkt-scraper/2025/clubs.j...,https://www.transfermarkt.co.uk/us-lecce/start...


In [4]:
clubs_info = clubs[[
    "club_id",
    "name",
    "domestic_competition_id",
    "squad_size",
    "average_age",
    "foreigners_percentage",
    "national_team_players",
    "stadium_seats"
]].copy()

clubs_info = clubs_info.rename(columns={
    "club_id": "current_club_id",
    "name": "club_name"
})

competitions_info = competitions[[
    "competition_id",
    "name",
    "country_name",
    "type",
    "sub_type",
    "confederation",
    "total_clubs"
]].copy()

competitions_info = competitions_info.rename(columns={
    "name": "competition_name",
    "type": "competition_type",
    "sub_type": "competition_sub_type"
})

club_league_info = clubs_info.merge(
    competitions_info,
    left_on="domestic_competition_id",
    right_on="competition_id",
    how="left"
)

club_league_info.head()

,current_club_id,club_name,domestic_competition_id,squad_size,average_age,foreigners_percentage,national_team_players,stadium_seats,competition_id,competition_name,country_name,competition_type,competition_sub_type,confederation,total_clubs
0,10,Arminia Bielefeld,L1,27,25.3,55.6,4,26515,L1,bundesliga,Germany,domestic_league,first_tier,europa,18.0
1,10004,Paris Football Club,FR1,31,28.5,54.8,8,19904,FR1,ligue-1,France,domestic_league,first_tier,europa,18.0
2,10010,Esporte Clube Bahia,BRA1,32,26.2,18.8,3,47364,BRA1,campeonato-brasileiro-serie-a,Brazil,domestic_league,first_tier,amerika,20.0
3,1003,Leicester City,GB1,29,25.9,58.6,9,32259,GB1,premier-league,England,domestic_league,first_tier,europa,20.0
4,1005,Unione Sportiva Lecce,IT1,27,25.1,85.2,10,31559,IT1,serie-a,Italy,domestic_league,first_tier,europa,20.0


In [5]:
extra_cols = [
    "club_name",
    "domestic_competition_id",
    "competition_id",
    "competition_name",
    "country_name",
    "competition_type",
    "competition_sub_type",
    "confederation",
    "total_clubs",
    "squad_size",
    "average_age",
    "foreigners_percentage",
    "national_team_players",
    "stadium_seats"
]

results_df=results_df.drop(
    columns=[cols for cols in extra_cols if cols in results_df.columns],
    errors="ignore"
)

undervalued_players=undervalued_players.drop(
    columns=[cols for cols in extra_cols if cols in undervalued_players.columns],
    errors="ignore"
)

results_df=results_df.merge(
    club_league_info,
    on="current_club_id",
    how="left"
)

undervalued_players=undervalued_players.merge(
    club_league_info,
    on="current_club_id",
    how="left"
)

In [6]:
undervalued_players[[
    "name",
    "age",
    "position",
    "club_name",
    "competition_name",
    "country_name",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur",
    "scouting_score"
]].head(10)

,name,age,position,club_name,competition_name,country_name,actual_market_value_eur,predicted_market_value_eur,value_gap_eur,scouting_score
0,Savinho,22,Attack,Manchester City Football Club,premier-league,England,40000000,6.259201e+07,2.259201e+07,0.934948
1,Gonçalo Ramos,24,Attack,Paris Saint-Germain Football Club,ligue-1,France,35000000,7.478560e+07,3.978560e+07,0.930623
2,Arsen Zakharyan,22,Midfield,Real Sociedad de Fútbol S.A.D.,laliga,Spain,10000000,1.737919e+07,7.379188e+06,0.913495
3,Rodrygo,25,Attack,Real Madrid Club de Fútbol,laliga,Spain,50000000,7.032666e+07,2.032666e+07,0.903633
4,Santiago Gimenez,25,Attack,Associazione Calcio Milan,serie-a,Italy,20000000,3.453081e+07,1.453081e+07,0.892388
5,Sem Steijn,24,Midfield,Feyenoord Rotterdam,eredivisie,Netherlands,11000000,1.696652e+07,5.966516e+06,0.891869
6,Roony Bardghji,20,Attack,Futbol Club Barcelona,laliga,Spain,15000000,4.144691e+07,2.644691e+07,0.889965
7,Clement Bischoff,20,Attack,Fußballclub Red Bull Salzburg,bundesliga,Austria,5000000,1.376778e+07,8.767778e+06,0.859862
8,Kang-in Lee,25,Midfield,Paris Saint-Germain Football Club,ligue-1,France,28000000,4.698373e+07,1.898373e+07,0.857093
9,Bjorn Meijer,23,Defender,Club Brugge Koninklijke Voetbalvereniging,jupiler-pro-league,Belgium,3500000,1.349758e+07,9.997577e+06,0.842907


In [7]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

results_df.to_csv(
    PROCESSED_DIR / "model_predictions.csv",
    index=False
)

undervalued_players.to_csv(
    PROCESSED_DIR / "undervalued_players.csv",
    index=False
)

print("Updated enriched CSV files saved successfully.")

Updated enriched CSV files saved successfully.


In [8]:
strength_cols = [
    "club_name",
    "competition_name",
    "country_name",
    "squad_size",
    "average_age",
    "foreigners_percentage",
    "national_team_players",
    "stadium_seats",
    "competition_type",
    "competition_sub_type",
    "total_clubs"
]

results_df[strength_cols].isnull().sum()

club_name                 63
competition_name          63
country_name              63
squad_size                63
average_age              340
foreigners_percentage    484
national_team_players     63
stadium_seats             63
competition_type          63
competition_sub_type      63
total_clubs               63
dtype: int64

In [13]:
club_numeric_cols = [
    "squad_size",
    "average_age",
    "foreigners_percentage",
    "national_team_players",
    "stadium_seats",
    "total_clubs"
]

for col in club_numeric_cols:
    if col in results_df.columns:
        results_df[col] = results_df[col].fillna(results_df[col].median())
    
    if col in undervalued_players.columns:
        undervalued_players[col] = undervalued_players[col].fillna(undervalued_players[col].median())


text_cols = [
    "club_name",
    "competition_name",
    "country_name",
    "competition_type",
    "competition_sub_type",
    "confederation"
]

for col in text_cols:
    if col in results_df.columns:
        results_df[col] = results_df[col].fillna("Unknown")
    
    if col in undervalued_players.columns:
        undervalued_players[col] = undervalued_players[col].fillna("Unknown")

In [14]:
results_df[strength_cols].isnull().sum()


club_name                0
competition_name         0
country_name             0
squad_size               0
average_age              0
foreigners_percentage    0
national_team_players    0
stadium_seats            0
competition_type         0
competition_sub_type     0
total_clubs              0
dtype: int64

In [21]:
clubs.head(1)

,club_id,club_code,name,domestic_competition_id,total_market_value,squad_size,average_age,foreigners_number,foreigners_percentage,national_team_players,stadium_name,stadium_seats,net_transfer_record,coach_name,last_season,filename,url
0,10,arminia-bielefeld,Arminia Bielefeld,L1,NaN,27,25.3,15,55.6,4,SchücoArena,26515,+€5.90m,NaN,2021,../data/raw/transfermarkt-scraper/2021/clubs.j...,https://www.transfermarkt.co.uk/arminia-bielef...


In [27]:
print(clubs["squad_size"].describe())

count    796.000000
mean      26.464824
std        7.650216
min        0.000000
25%       25.000000
50%       28.000000
75%       30.000000
max       52.000000
Name: squad_size, dtype: float64


In [28]:
print(clubs.loc[clubs["squad_size"].idxmax()])

club_id                                                                 6502
club_code                                             jeonbuk-hyundai-motors
name                                                  Jeonbuk Hyundai Motors
domestic_competition_id                                                 RSK1
total_market_value                                                       NaN
squad_size                                                                52
average_age                                                             24.5
foreigners_number                                                          6
foreigners_percentage                                                   11.5
national_team_players                                                      4
stadium_name                                        Jeonju World Cup Stadium
stadium_seats                                                          50000
net_transfer_record                                                  +€1.69m

In [30]:
import pandas as pd
import numpy as np

# -------------------------------
# CLUB STRENGTH SCORE - CLEAN VERSION
# -------------------------------

# 1. Select club strength columns
club_strength_cols = [
    "national_team_players",
    "stadium_seats",
    "foreigners_percentage",
    "squad_size"
]

# 2. Fill missing values using median
for col in club_strength_cols:
    results_df[col] = results_df[col].fillna(results_df[col].median())
    undervalued_players[col] = undervalued_players[col].fillna(undervalued_players[col].median())

# 3. Min-Max normalization function
def min_max_normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# 4. Create normalized score columns for results_df
for col in club_strength_cols:
    results_df[col + "_score"] = min_max_normalize(results_df[col])

# 5. Create normalized score columns for undervalued_players
for col in club_strength_cols:
    undervalued_players[col + "_score"] = min_max_normalize(undervalued_players[col])

# 6. Create final club strength score
results_df["club_strength_score"] = (
    0.40 * results_df["national_team_players_score"] +
    0.30 * results_df["stadium_seats_score"] +
    0.20 * results_df["foreigners_percentage_score"] +
    0.10 * results_df["squad_size_score"]
)

undervalued_players["club_strength_score"] = (
    0.40 * undervalued_players["national_team_players_score"] +
    0.30 * undervalued_players["stadium_seats_score"] +
    0.20 * undervalued_players["foreigners_percentage_score"] +
    0.10 * undervalued_players["squad_size_score"]
)

# 7. Check top clubs by strength score
results_df[[
    "club_name",
    "competition_name",
    "country_name",
    "national_team_players",
    "stadium_seats",
    "foreigners_percentage",
    "squad_size",
    "club_strength_score"
]].sort_values("club_strength_score", ascending=False).head(2)

,club_name,competition_name,country_name,national_team_players,stadium_seats,foreigners_percentage,squad_size,club_strength_score
4910,Real Madrid Club de Fútbol,laliga,Spain,18.0,83186.0,60.7,28.0,0.800586
4374,Real Madrid Club de Fútbol,laliga,Spain,18.0,83186.0,60.7,28.0,0.800586


In [34]:
league_strength_map = {
    "premier-league": 1.00,
    "laliga": 0.96,
    "serie-a": 0.92,
    "bundesliga": 0.91,
    "ligue-1": 0.87,

    "liga-portugal": 0.78,
    "eredivisie": 0.76,
    "campeonato-brasileiro-serie-a": 0.74,
    "major-league-soccer": 0.70,
    "saudi-pro-league": 0.69,
    "jupiler-pro-league": 0.67,
    "super-lig": 0.66,
    "scottish-premiership": 0.64,

    "premier-liga": 0.62,
    "super-league-1": 0.61,
    "superliga": 0.60,
    "super-league": 0.59,
    "liga-mx-clausura": 0.58,
    "j1-league": 0.56,
    "k-league-1": 0.54,

    "pko-bp-ekstraklasa": 0.52,
    "chance-liga": 0.51,
    "allsvenskan": 0.50,
    "eliteserien": 0.49,
    "super-liga-srbije": 0.48,
    "supersport-hnl": 0.47,
    "a-league-men": 0.46,
    "torneo-apertura": 0.45,
    "liga-betplay-dimayor": 0.44,

    "unknown": 0.50
}

results_df["league_strength_score"] = (
    results_df["competition_name"]
    .map(league_strength_map)
    .fillna(0.50)
)

undervalued_players["league_strength_score"] = (
    undervalued_players["competition_name"]
    .map(league_strength_map)
    .fillna(0.50)
)

In [35]:
results_df[["competition_name", "league_strength_score"]] \
    .drop_duplicates() \
    .sort_values("league_strength_score", ascending=False)

,competition_name,league_strength_score
14,premier-league,1.00
33,laliga,0.96
23,serie-a,0.92
30,bundesliga,0.91
0,ligue-1,0.87
10,liga-portugal,0.78
13,eredivisie,0.76
32,campeonato-brasileiro-serie-a,0.74
387,major-league-soccer,0.70
15,saudi-pro-league,0.69


In [36]:
# -------------------------------
# GRADIENT BOOSTING V4
# Clean club + league strength model
# -------------------------------

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1. Create V4 feature dataset
features_df_v4 = results_df[[
    "age",
    "minutes_played",
    "goals",
    "assists",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "club_strength_score",
    "league_strength_score",
    "position",
    "log_market_value"
]].copy()

# 2. One-hot encode position
features_df_v4 = pd.get_dummies(
    features_df_v4,
    columns=["position"],
    drop_first=True
)

# 3. Split X and y
X_v4 = features_df_v4.drop("log_market_value", axis=1)
y_v4 = features_df_v4["log_market_value"]

# 4. Train-test split
X_train_v4, X_test_v4, y_train_v4, y_test_v4 = train_test_split(
    X_v4,
    y_v4,
    test_size=0.2,
    random_state=42
)

# 5. Models
models_v4 = {
    "Linear Regression V4": LinearRegression(),
    "Random Forest V4": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting V4": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

# 6. Train and evaluate
v4_results = []

for model_name, model in models_v4.items():
    model.fit(X_train_v4, y_train_v4)
    
    y_pred_v4 = model.predict(X_test_v4)
    
    mae_v4 = mean_absolute_error(y_test_v4, y_pred_v4)
    rmse_v4 = np.sqrt(mean_squared_error(y_test_v4, y_pred_v4))
    r2_v4 = r2_score(y_test_v4, y_pred_v4)
    
    v4_results.append({
        "model": model_name,
        "mae": mae_v4,
        "rmse": rmse_v4,
        "r2_score": r2_v4
    })

# 7. Show results
v4_model_comparison = pd.DataFrame(v4_results).sort_values(
    by="r2_score",
    ascending=False
)

v4_model_comparison

,model,mae,rmse,r2_score
2,Gradient Boosting V4,0.674879,0.864020,0.704390
1,Random Forest V4,0.677945,0.865309,0.703507
0,Linear Regression V4,0.918854,1.149052,0.477181


In [37]:
# Get the trained Gradient Boosting V4 model
gb_v4_model = models_v4["Gradient Boosting V4"]

# Create feature importance table
v4_feature_importance = pd.DataFrame({
    "feature": X_v4.columns,
    "importance": gb_v4_model.feature_importances_
})

# Sort highest importance first
v4_feature_importance = v4_feature_importance.sort_values(
    by="importance",
    ascending=False
)

v4_feature_importance

,feature,importance
1,minutes_played,0.290311
0,age,0.276028
7,club_strength_score,0.242263
8,league_strength_score,0.111514
2,goals,0.026813
3,assists,0.021494
5,assists_per_90,0.018861
6,goal_contributions_per_90,0.010023
4,goals_per_90,0.001784
10,position_Goalkeeper,0.000605


In [38]:
results_df.to_csv(PROCESSED_DIR / "model_predictions.csv", index=False)
undervalued_players.to_csv(PROCESSED_DIR / "undervalued_players.csv", index=False)

print("Updated prediction and undervalued player files saved.")

Updated prediction and undervalued player files saved.


In [39]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save V4 model comparison
v4_model_comparison.to_csv(
    PROCESSED_DIR / "v4_model_comparison.csv",
    index=False
)

# Save V4 feature importance
v4_feature_importance.to_csv(
    PROCESSED_DIR / "v4_feature_importance.csv",
    index=False
)

# Save updated predictions with club_strength_score and league_strength_score
results_df.to_csv(
    PROCESSED_DIR / "model_predictions.csv",
    index=False
)

# Save updated undervalued players with club_strength_score and league_strength_score
undervalued_players.to_csv(
    PROCESSED_DIR / "undervalued_players.csv",
    index=False
)

print("V4 files saved successfully.")

V4 files saved successfully.


In [41]:
results_df.to_csv(
    PROCESSED_DIR / "model_predictions.csv",
    index=False
)

undervalued_players.to_csv(
    PROCESSED_DIR / "undervalued_players.csv",
    index=False
)

print("Main prediction and undervalued player files updated.")

Main prediction and undervalued player files updated.


In [42]:
check_df = pd.read_csv(PROCESSED_DIR / "model_predictions.csv")

check_df[[
    "club_name",
    "competition_name",
    "country_name",
    "club_strength_score",
    "league_strength_score"
]].head()

,club_name,competition_name,country_name,club_strength_score,league_strength_score
0,Racing Club de Strasbourg Alsace,ligue-1,France,0.560696,0.87
1,Ural Yekaterinburg,premier-liga,Russia,0.261993,0.62
2,U Craiova 1948 Club Sportiv SA,superliga,Romania,0.381286,0.60
3,IK Sirius,allsvenskan,Sweden,0.190554,0.50
4,Trabzonspor Kulübü,super-lig,Türkiye,0.488181,0.66
